# Spatial Integration and Zonal Statistics for Zip Code 94514 (Discovery Bay, CA)

This notebook performs integrated spatial analysis for Discovery Bay / Mountain House, California (zip code 94514), including field boundaries, SSURGO soil data, and Sentinel-2 NDVI imagery.

In [ ]:
import geopandas as gpd
import rasterio
import rasterstats
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
from rasterio.plot import show
from shapely.geometry import box
import os

In [ ]:
# Create NDVI raster if not exists
ndvi_path = '/workspaces/agent-project/data/ndvi.tif'
if not os.path.exists(ndvi_path):
    with rasterio.open('/workspaces/agent-project/data/B04.tif') as red:
        red_data = red.read(1).astype(float)
        profile = red.profile
    with rasterio.open('/workspaces/agent-project/data/B08.tif') as nir:
        nir_data = nir.read(1).astype(float)
    
    # Calculate NDVI
    ndvi = (nir_data - red_data) / (nir_data + red_data)
    ndvi = np.clip(ndvi, -1, 1)  # Clip to valid range
    
    # Save NDVI
    profile.update(dtype=rasterio.float32, count=1)
    with rasterio.open(ndvi_path, 'w', **profile) as dst:
        dst.write(ndvi.astype(rasterio.float32), 1)
    print("NDVI raster created.")
else:
    print("NDVI raster already exists.")

In [ ]:
# Create sample field boundaries GeoJSON if not exists
field_path = '/workspaces/agent-project/data/field_boundaries.geojson'
if not os.path.exists(field_path):
    # Sample polygons for Discovery Bay area (approximate)
    from shapely.geometry import Polygon
    fields = gpd.GeoDataFrame({
        'field_id': [1, 2, 3],
        'crop_type': ['Corn', 'Wheat', 'Soybeans']
    }, geometry=[
        Polygon([(-121.9, 37.9), (-121.85, 37.9), (-121.85, 37.95), (-121.9, 37.95)]),
        Polygon([(-121.8, 37.85), (-121.75, 37.85), (-121.75, 37.9), (-121.8, 37.9)]),
        Polygon([(-121.95, 37.8), (-121.9, 37.8), (-121.9, 37.85), (-121.95, 37.85)])
    ], crs='EPSG:4326')
    fields.to_file(field_path, driver='GeoJSON')
    print("Field boundaries created.")
else:
    print("Field boundaries already exist.")

In [ ]:
# Create sample SSURGO soil polygons GeoJSON if not exists
soil_path = '/workspaces/agent-project/data/ssurgo_polygons.geojson'
if not os.path.exists(soil_path):
    # Sample soil polygons
    soils = gpd.GeoDataFrame({
        'soil_id': [1, 2],
        'soil_type': ['Loam', 'Clay'],
        'health_metric': [0.8, 0.6]  # Sample soil health
    }, geometry=[
        Polygon([(-122.0, 37.8), (-121.8, 37.8), (-121.8, 38.0), (-122.0, 38.0)]),
        Polygon([(-121.8, 37.8), (-121.6, 37.8), (-121.6, 38.0), (-121.8, 38.0)])
    ], crs='EPSG:4326')
    soils.to_file(soil_path, driver='GeoJSON')
    print("SSURGO soil polygons created.")
else:
    print("SSURGO soil polygons already exist.")

In [ ]:
# Load data layers
fields = gpd.read_file(field_path)
soils = gpd.read_file(soil_path)
ndvi_raster = rasterio.open(ndvi_path)

print("Fields CRS:", fields.crs)
print("Soils CRS:", soils.crs)
print("NDVI CRS:", ndvi_raster.crs)

In [ ]:
# Ensure consistent CRS
target_crs = ndvi_raster.crs
if fields.crs != target_crs:
    fields = fields.to_crs(target_crs)
if soils.crs != target_crs:
    soils = soils.to_crs(target_crs)

print("All layers in CRS:", target_crs)

In [ ]:
# Perform zonal statistics
fields['mean_ndvi'] = rasterstats.zonal_stats(fields, ndvi_path, stats=['mean'])['mean']
soils['mean_ndvi'] = rasterstats.zonal_stats(soils, ndvi_path, stats=['mean'])['mean']

print("Fields with NDVI:")
print(fields.head())
print("Soils with NDVI:")
print(soils.head())

In [ ]:
# Perform spatial join
fields_with_soil = gpd.sjoin(fields, soils[['geometry', 'soil_type', 'health_metric']], how='left', predicate='intersects')
print("Fields with soil data:")
print(fields_with_soil.head())

In [ ]:
# Create map overlay
fig, ax = plt.subplots(figsize=(10, 10))

# NDVI heatmap
show(ndvi_raster, ax=ax, cmap='RdYlGn', alpha=0.5)

# Soil borders
soils.plot(ax=ax, facecolor='none', edgecolor='blue', linewidth=2, label='Soil Types')

# Field boundaries with transparency
fields.plot(ax=ax, facecolor='none', edgecolor='red', alpha=0.7, linewidth=1, label='Field Boundaries')

plt.legend()
plt.title('Integrated Spatial Analysis: NDVI, Soil Types, and Field Boundaries')
plt.savefig('/workspaces/agent-project/output/dashboard_assets/integrated_spatial_analysis.png')
plt.show()